Kaarina Mattika, CS87A, Section 1758
https://github.com/kmattika/cs82a-portfolio

In [1]:
import pandas as pd
import seaborn as sns

In [2]:
df = pd.read_csv('C:\Users\kamat\OneDrive\Desktop\CS82A\messy_sales.csv')

Step One

In [3]:
display(df.head(10))
display(df.shape)
display(df.info())
display(df.describe())

,order_id,date,product,price,qty,zip
0,1254,04/06/2026,mug,11.36,4,60614
1,1057,2026-05-30,charger,14.79,3,30303
2,1150,05/06/2026,mug,5.50,1,10001
3,1066,05/30/2026,notebook,NaN,1,98101
4,1114,04/06/2026,webcam,45.59,4,90405
5,1280,04/22/2026,pen set,NaN,4,30303
6,1204,04/18/2026,charger,8.41,2,2134
7,1249,2026-05-15,keyboard,75.04,4,90405
8,1037,2026-05-19,mug,9.31,2,60614
9,1177,2026-05-13,mug,10.11,3,90210


(300, 6)

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  300 non-null    int64  
 1   date      300 non-null    str    
 2   product   300 non-null    str    
 3   price     288 non-null    float64
 4   qty       300 non-null    int64  
 5   zip       300 non-null    int64  
dtypes: float64(1), int64(3), str(2)
memory usage: 19.2 KB


None

,order_id,price,qty,zip
count,300.000000,288.000000,300.000000,300.000000
mean,1144.816667,56.068438,2.913333,45385.276667
std,84.515319,72.307182,1.560107,39345.608641
min,1000.000000,2.930000,-5.000000,2116.000000
25%,1071.750000,10.945000,2.000000,2134.000000
50%,1144.500000,37.530000,3.000000,30303.000000
75%,1217.250000,61.417500,4.000000,90210.000000
max,1291.000000,479.290000,5.000000,98101.000000


Step Two

Three early problems I can identify: there are twelve fewer entries in the 'price' column than other columns, the 'date' column has non-standard formatting, and there is at minimum one typo on the 'zip' column where one zip code, typically containing five digits, contains four digits.

Step Three

In [4]:
df.isna().sum()

order_id     0
date         0
product      0
price       12
qty          0
zip          0
dtype: int64

Step Four

In [5]:
fill_value = df['price'].median()
df['price'] = df['price'].fillna(fill_value)

I chose to use the median, which avoids the input of extreme price variations. Given the sizable price fluxtuations between products, the mean is not a suitable statistical instrument here. Ideally, we could manually correct these missing values with a price lookup tool or by matching an SKU to a catalogue, but the prompt specifically instructed the use of a statitical instrument.

Step Five

In [6]:
print(df.duplicated().sum())
df = df.drop_duplicates()
display(df.shape)

8


(292, 6)

Step Six

In [7]:
df['zip'] = df['zip'].astype(str).str.zfill(5)

Step Seven

In [8]:
df['date'] = pd.to_datetime(df['date'], format='mixed')
display(df.dtypes)

order_id             int64
date        datetime64[us]
product                str
price              float64
qty                  int64
zip                    str
dtype: object

Step Eight

In [9]:
df[df['qty'] < 0]

,order_id,date,product,price,qty,zip
202,1140,2026-04-18,webcam,38.69,-5,98101
262,1233,2026-05-09,desk lamp,22.64,-4,10001
297,1025,2026-04-27,keyboard,47.30,-2,02116


In [10]:
display(df.groupby('product')['price'].median())
display(df.groupby('product')['qty'].median())

product
backpack       41.305
charger        17.170
desk lamp      20.710
headphones     53.665
keyboard       46.195
monitor       207.400
mug             7.470
notebook        6.580
pen set        10.535
webcam         67.460
Name: price, dtype: float64

product
backpack      3.0
charger       3.0
desk lamp     3.0
headphones    3.0
keyboard      3.0
monitor       3.0
mug           3.0
notebook      3.0
pen set       3.0
webcam        3.0
Name: qty, dtype: float64

Running some quick analytics on the numbers present for these data errors makes me believe that these are most likely a set of typos in the 'qty' column. The median price and quantity for the 'desk lamp' and 'keyboard' appear to be very closely related to the rows with errors present (262 and 297), while the 'webcam' median price and quantity are a bit further away compared to row 202. We could do additional analytics to determine if the data values for row 202 are within one or two standard deviations from the median 'webcam' value in 'price' and 'qty', I'm sufficiently satisfied with the results to apply the following cleanups: change the 'qty' value in rows 202, 262, and 297 to their absolute values (5, 4, and 2, respectively).

In [11]:
df.loc[202, 'qty'] = 5;
df.loc[262, 'qty'] = 4;
df.loc[297, 'qty'] = 2
df[df['qty'] < 0]

,order_id,date,product,price,qty,zip


Step Nine

In [12]:
df.to_csv(r'C:\Users\kamat\OneDrive\Desktop\CS82A\sales_clean.csv', index=False)

Step Ten

Cleaning log:

Three hundred rows were originally loaded via "C:\Users\kamat\OneDrive\Desktop\CS82A\messy_sales.csv"
Filled twelve 'price' column values with the median value of the remaining 288 values in the same column
Eight duplicate values were removed, leaving two hundred and eighty eight rows remaining
'Zip' column data type changed from int64 to string to restore leading zeroes in five-digit zip codes; original format produced four-digit zip code errors
Changed the 'qty' values in rows 202, 262, and 297 to their absolute value to correct what I believe is a data input error.

Incorrectly cleaning the data in any way can produce skewed or inaccurate results, or even a flawed methodology, for evaluating sales data, which would result in false insights being 'discovered' from faulty data and practices, and incorrect business decisions being applied that can result in fewer sales, higher dead inventory stocks, and incorrect pricing mechanisms being applied. For example - if we overestimate the demand for monitors and underestimate the demand for webcams, we could have a lot of monitors sittling idily in a warehouse, unsold due to prices higher than market demand, while our webcams sell out rapidly at an unsustainable price.